# 📈 Stock Market Predictor — LSTM Training
**Kelvin Kipkirui | DAC-01-0010/2025 | Zetech University**

---
**Before running this notebook:**
1. Upload these files from your `Data` folder to Colab:
   - `X_train_AAPL.npy`, `X_val_AAPL.npy`, `X_test_AAPL.npy`
   - `y_train_AAPL.npy`, `y_val_AAPL.npy`, `y_test_AAPL.npy`
   - `scaler_AAPL.pkl`, `feature_cols_AAPL.pkl`
2. Make sure **GPU is enabled**: Runtime → Change runtime type → T4 GPU
3. Run all cells in order.

## 📦 CELL 1: Install & Import

In [ ]:
!pip install tensorflow joblib --quiet

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📥 CELL 2: Upload & Load Data

In [ ]:
from google.colab import files
print('Upload your .npy and .pkl files from your Data folder:')
uploaded = files.upload()

In [ ]:
TICKER = 'AAPL'

X_train = np.load(f'X_train_{TICKER}.npy')
X_val   = np.load(f'X_val_{TICKER}.npy')
X_test  = np.load(f'X_test_{TICKER}.npy')
y_train = np.load(f'y_train_{TICKER}.npy')
y_val   = np.load(f'y_val_{TICKER}.npy')
y_test  = np.load(f'y_test_{TICKER}.npy')

scaler       = joblib.load(f'scaler_{TICKER}.pkl')
feature_cols = joblib.load(f'feature_cols_{TICKER}.pkl')

print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print('✅ Data loaded.')

## 🧠 CELL 3: Build LSTM Model

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])
print(f'Input shape: {input_shape}  (timesteps=60, features=19)')

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=input_shape),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

## 🏋️ CELL 4: Train LSTM

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

print('Training LSTM (max 100 epochs, early stopping patience=10)...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print(f'\n✅ Training complete.')
print(f'   Epochs trained  : {len(history.history["loss"])}')
print(f'   Final train loss: {history.history["loss"][-1]:.6f}')
print(f'   Final val loss  : {history.history["val_loss"][-1]:.6f}')

## 📊 CELL 5: Evaluate on Test Set

In [ ]:
def inverse_transform_price(scaled_values, scaler, feature_cols):
    close_idx = feature_cols.index('Close')
    dummy = np.zeros((len(scaled_values), len(feature_cols)))
    dummy[:, close_idx] = scaled_values
    return scaler.inverse_transform(dummy)[:, close_idx]

y_pred_scaled = model.predict(X_test, verbose=0).flatten()
y_pred_real   = inverse_transform_price(y_pred_scaled, scaler, feature_cols)
y_true_real   = inverse_transform_price(y_test, scaler, feature_cols)

mae     = mean_absolute_error(y_true_real, y_pred_real)
rmse    = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
r2      = r2_score(y_true_real, y_pred_real)
dir_acc = np.mean((np.diff(y_true_real) > 0) == (np.diff(y_pred_real) > 0)) * 100

print('='*50)
print(f'  LSTM RESULTS — {TICKER} TEST SET')
print('='*50)
print(f'  MAE                 : ${mae:.2f}')
print(f'  RMSE                : ${rmse:.2f}')
print(f'  R² Score            : {r2:.4f}')
print(f'  Directional Accuracy: {dir_acc:.1f}%')
print('='*50)

## 📈 CELL 6: Plot Results

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Actual vs Predicted
x = range(len(y_true_real))
axes[0].plot(x, y_true_real, color='#1F4E79', lw=1.5, label='Actual Price')
axes[0].plot(x, y_pred_real, color='#2E75B6', lw=1.5,
             linestyle='--', label='LSTM Prediction')
axes[0].fill_between(x, y_true_real, y_pred_real, alpha=0.08, color='#2E75B6')
axes[0].set_title(f'{TICKER} — LSTM: Actual vs Predicted Price',
                  fontweight='bold', fontsize=13)
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

# Loss curves
epochs = range(1, len(history.history['loss']) + 1)
axes[1].plot(epochs, history.history['loss'],
             color='#1F4E79', lw=2, label='Training Loss')
axes[1].plot(epochs, history.history['val_loss'],
             color='#E74C3C', lw=2, linestyle='--', label='Validation Loss')
axes[1].set_title(f'{TICKER} — LSTM Training & Validation Loss',
                  fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (MSE)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{TICKER}_lstm_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved.')

## 💾 CELL 7: Save & Download Model

In [ ]:
import zipfile

# Save model
model.save(f'lstm_model_{TICKER}')
print(f'✅ Model saved: lstm_model_{TICKER}/')

# Zip model folder for download
zip_name = f'lstm_model_{TICKER}.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in __import__('os').walk(f'lstm_model_{TICKER}'):
        for file in files:
            filepath = __import__('os').path.join(root, file)
            zipf.write(filepath)

# Download everything
files_to_download = [
    zip_name,
    f'{TICKER}_lstm_results.png'
]

for f in files_to_download:
    files.download(f)
    print(f'⬇️  Downloading: {f}')

print('\n✅ All done!')
print('   Unzip lstm_model_AAPL.zip into your Saved Models folder.')